# Day 5: Teaching the Model About Your Documents

**LangChain Hands-on Series | Day 5 of 10**
Trainer: Ajeetkumar

---

Ask ChatGPT "How many days of annual leave do NovaTech employees get?" and it has no idea. NovaTech is a made-up company, and its HR policy is a private document the model has never seen.

This is the situation in almost every company project: the useful knowledge lives in **your** PDFs, wikis and spreadsheets. Today we build the foundation for using them. Tomorrow (Day 6) we connect it to the LLM - that combination is called **RAG**.

Today is the "library" part of the story:

```
   1. LOAD          2. SPLIT             3. EMBED                 4. STORE              5. SEARCH
 +---------+     +-----------+      +------------------+      +--------------+      +-------------------+
 | PDF/TXT | --> |  chunks   | -->  | numbers (vectors)| -->  | vector store | <--  | "how many leaves?"|
 |  CSV/MD |     | of ~500   |      | that capture     |      |  (the index) |      |  -> top 3 chunks  |
 +---------+     | characters|      | meaning          |      +--------------+      +-------------------+
                 +-----------+      +------------------+
```

A library analogy helps: **loading** is receiving books, **splitting** is cutting them into index cards, **embedding** is writing a "meaning code" on each card, **storing** is filing the cards, and **searching** is the librarian finding the cards closest in meaning to your question.

**Mini project:** semantic search over the policies of NovaTech, our fictional company. The files are in the `data/` folder.

## Setup

In [ ]:
import os
import warnings
from dotenv import load_dotenv

warnings.filterwarnings("ignore")        # hide noisy deprecation warnings from some loaders
load_dotenv()
MODEL_NAME = os.getenv("OPENAI_MODEL", "gpt-4.1-nano")

os.listdir("data/policies")              # the documents we will work with today

## 1. The Document object

Everything LangChain loads becomes a `Document`. It is a very simple object with two parts:

- `page_content` - the text
- `metadata` - a dictionary with extra information (file name, page number, author, anything you like)

Metadata looks unimportant at first, but it is what lets you later say "this answer came from page 2 of the Code of Conduct" or "only search HR documents".

In [ ]:
from langchain_core.documents import Document

doc = Document(
    page_content="Employees receive 24 days of paid annual leave per year.",
    metadata={"source": "leave_policy.txt", "department": "HR"},
)

print(doc.page_content)
print(doc.metadata)

## 2. Loading files

LangChain has loaders for hundreds of formats (PDF, Word, web pages, Notion, S3...). Every loader returns a **list of Documents**, so the rest of your code never cares what format the file was.

### A text file

In [ ]:
from langchain_community.document_loaders import TextLoader

leave_docs = TextLoader("data/policies/leave_policy.txt", encoding="utf-8").load()

print("Documents loaded:", len(leave_docs))
print("Metadata:", leave_docs[0].metadata)
print(leave_docs[0].page_content[:300])

### A whole folder at once

In [ ]:
from langchain_community.document_loaders import DirectoryLoader

folder_docs = DirectoryLoader(
    "data/policies",
    glob="*.txt",                           # only .txt files
    loader_cls=TextLoader,                  # how to open each file
    loader_kwargs={"encoding": "utf-8"},
).load()

for d in folder_docs:
    print(d.metadata["source"], "-", len(d.page_content), "characters")

### A PDF

`PyPDFLoader` creates **one Document per page** and records the page number in the metadata. Very handy for citations later.

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

pdf_docs = PyPDFLoader("data/policies/code_of_conduct.pdf").load()

print("Pages:", len(pdf_docs))
for p in pdf_docs:
    print("page", p.metadata["page"], "->", p.page_content[:80].replace("\n", " "), "...")

### A CSV file

`CSVLoader` turns **each row** into its own Document. Good for FAQs or product catalogues.

In [ ]:
from langchain_community.document_loaders import CSVLoader

order_docs = CSVLoader("data/orders.csv", encoding="utf-8").load()

print("Rows loaded:", len(order_docs))
print(order_docs[0].page_content)

Let's collect all the policy documents (text files, the markdown FAQ and the PDF) into one list. We also add a simple `"doc_type"` field to every document's metadata - we will filter on it later.

In [ ]:
md_docs = TextLoader("data/policies/product_faq.md", encoding="utf-8").load()

all_docs = folder_docs + md_docs + pdf_docs

for d in all_docs:
    d.metadata["file"] = os.path.basename(d.metadata["source"])
    d.metadata["doc_type"] = "product" if "faq" in d.metadata["file"] else "policy"

print("Total documents:", len(all_docs))

## 3. Splitting into chunks

Why not just store each whole file? Three reasons:

1. **Precision.** If a question is about sick leave, we want the paragraph about sick leave, not a whole 5-page policy.
2. **Cost.** Whatever we find gets sent to the LLM. Smaller pieces mean fewer tokens.
3. **Limits.** Embedding models have an input size limit.

Picture cutting a long article into index cards. Two settings matter:

- `chunk_size` - how big each card is (in characters)
- `chunk_overlap` - how much each card repeats from the end of the previous one, so a sentence cut in half is not lost

```
 Text:   |-------------------------------------------------------------|
 Chunk 1 |================|
 Chunk 2            |================|          <- the overlap is the shared part
 Chunk 3                       |================|
```

### See it with a tiny example

In [ ]:
from langchain_text_splitters import CharacterTextSplitter

sample_text = "abcdefghijklmnopqrstuvwxyz"

tiny_splitter = CharacterTextSplitter(chunk_size=10, chunk_overlap=3, separator="")
tiny_splitter.split_text(sample_text)

**Pause and predict:** look at the output. Can you spot the 3 letters each chunk repeats from the previous one?

### The splitter you will use 90% of the time

`RecursiveCharacterTextSplitter` tries to split at natural boundaries first - paragraphs, then lines, then sentences, then words - and only cuts mid-word as a last resort. This keeps chunks readable.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=50)

chunks = splitter.split_documents(all_docs)      # split_documents keeps the metadata on every chunk

print("Documents:", len(all_docs), "-> chunks:", len(chunks))
print("\nExample chunk:\n", chunks[3].page_content)
print("\nIts metadata:", chunks[3].metadata)

In [ ]:
import matplotlib.pyplot as plt

lengths = [len(c.page_content) for c in chunks]

plt.figure(figsize=(6, 3))
plt.hist(lengths, bins=12, color="#4C72B0", edgecolor="white")
plt.axvline(400, color="red", linestyle="--", label="chunk_size = 400")
plt.xlabel("Characters per chunk")
plt.ylabel("Number of chunks")
plt.title("Chunk sizes after splitting")
plt.legend()
plt.show()

Notice no chunk goes over the red line, and many are smaller - the splitter preferred to stop at a paragraph break rather than fill every chunk to the maximum.

### How does chunk size change things?

There is no perfect chunk size. Small chunks are precise but can lose context; large chunks keep context but are less focused. 300 to 1000 characters is a common starting range. Let's see how the number of chunks changes:

In [ ]:
sizes = [200, 400, 800, 1600]
counts = [len(RecursiveCharacterTextSplitter(chunk_size=s, chunk_overlap=50).split_documents(all_docs)) for s in sizes]

plt.figure(figsize=(5, 3))
plt.bar([str(s) for s in sizes], counts, color="#DD8452")
plt.xlabel("chunk_size")
plt.ylabel("Number of chunks")
plt.title("Bigger chunks, fewer of them")
plt.show()

## 4. Embeddings - turning meaning into numbers

This is the most magical-feeling idea of the day, so let's go slowly.

An **embedding** is a list of numbers (a vector) that represents the *meaning* of a piece of text. The embedding model is trained so that texts with similar meaning get similar numbers.

A useful picture is a map. Imagine every sentence placed on a giant map where similar meanings sit close together: "annual leave" lands near "vacation days", far from "password rules". An embedding is the sentence's coordinates on that map - just with around 1500 coordinates instead of 2.

```
  "How many vacation days do I get?"   -->  [0.012, -0.044, 0.091, ... ]
  "Annual leave is 24 days per year"   -->  [0.010, -0.041, 0.088, ... ]   <- very close
  "Passwords must be 12 characters"    -->  [-0.07,  0.031, -0.02, ... ]   <- far away
```

Notice that the first two sentences share almost no words, yet their meaning is the same. Keyword search would miss this. Embeddings catch it.

In [ ]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")    # OpenAI's small, cheap embedding model

vector = embeddings.embed_query("How many vacation days do I get?")

print("Length of the vector:", len(vector))
print("First 8 numbers    :", [round(v, 4) for v in vector[:8]])

### Measuring "closeness": cosine similarity

To compare two vectors we use **cosine similarity**: 1.0 means the same direction (same meaning), around 0 means unrelated.

In [ ]:
import numpy as np

def cosine_similarity(a, b):
    a, b = np.array(a), np.array(b)
    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b)))

sentences = [
    "How many vacation days do I get?",
    "Annual leave is 24 days per year.",
    "Passwords must be at least 12 characters long.",
    "What is the minimum password length?",
    "The hotel limit in metro cities is INR 6,000.",
]

vectors = embeddings.embed_documents(sentences)      # embed several texts in one call

print("vacation vs annual leave :", round(cosine_similarity(vectors[0], vectors[1]), 3))
print("vacation vs password     :", round(cosine_similarity(vectors[0], vectors[2]), 3))

### Visualization: a similarity heatmap

Let's compare every sentence with every other sentence. Darker squares mean more similar meaning. You should see two "pairs" light up: the two leave sentences and the two password sentences.

In [ ]:
n = len(sentences)
sim_matrix = np.array([[cosine_similarity(vectors[i], vectors[j]) for j in range(n)] for i in range(n)])

short_labels = ["vacation days?", "annual leave 24", "password 12 chars", "min password?", "hotel limit"]

plt.figure(figsize=(6, 5))
plt.imshow(sim_matrix, cmap="Blues")
plt.colorbar(label="cosine similarity")
plt.xticks(range(n), short_labels, rotation=45, ha="right")
plt.yticks(range(n), short_labels)
for i in range(n):
    for j in range(n):
        plt.text(j, i, f"{sim_matrix[i, j]:.2f}", ha="center", va="center", fontsize=8)
plt.title("Which sentences mean similar things?")
plt.tight_layout()
plt.show()

### Visualization: our chunks on a 2D map

Our vectors have hundreds of dimensions, which we cannot draw. PCA is a technique that squashes them down to 2 dimensions while keeping as much of the structure as possible. Each dot below is a chunk, coloured by which file it came from. Chunks from the same policy tend to sit together.

In [ ]:
from sklearn.decomposition import PCA

chunk_vectors = embeddings.embed_documents([c.page_content for c in chunks])
points = PCA(n_components=2).fit_transform(np.array(chunk_vectors))

files = sorted({c.metadata["file"] for c in chunks})
plt.figure(figsize=(7, 5))
for f in files:
    idx = [i for i, c in enumerate(chunks) if c.metadata["file"] == f]
    plt.scatter(points[idx, 0], points[idx, 1], label=f, s=60, alpha=0.8)
plt.legend(fontsize=8)
plt.title("Each dot is a chunk - similar topics cluster together")
plt.xticks([]); plt.yticks([])
plt.show()

## 5. Vector stores - a database that searches by meaning

Comparing a question with every chunk ourselves works for 30 chunks, but not for 3 million. A **vector store** stores the vectors and finds the closest ones very quickly.

| Vector store | When to use it |
|--------------|----------------|
| `InMemoryVectorStore` | learning and small demos; lost when Python stops |
| `Chroma` | local projects; saves to disk |
| `FAISS` | very fast local search over large collections |
| Pinecone, Qdrant, Weaviate, pgvector, Azure AI Search | production, managed services |

They all share the same methods, so switching later is easy.

### InMemoryVectorStore

In [ ]:
from langchain_core.vectorstores import InMemoryVectorStore

vector_store = InMemoryVectorStore.from_documents(chunks, embedding=embeddings)   # embeds and stores every chunk

print("Stored chunks:", len(vector_store.store))

In [ ]:
results = vector_store.similarity_search("How many vacation days do I get?", k=3)

for i, r in enumerate(results, 1):
    print(f"--- Result {i} (from {r.metadata['file']}) ---")
    print(r.page_content[:200], "\n")

**Pause and predict:** the word "vacation" does not appear anywhere in our leave policy. Did the search still find the annual leave section? Why?

<details>
<summary>Click to see the explanation</summary>

It should, because the search compares meanings, not words. "Vacation days" and "annual leave" are close on the meaning map. This is the core advantage of semantic search over keyword search.
</details>

### Seeing the scores

`similarity_search_with_score` also returns how close each result is. Useful for deciding when results are "too far" to be trusted.

In [ ]:
scored = vector_store.similarity_search_with_score("Can I bring my own USB drive?", k=4)

for doc, score in scored:
    print(f"{score:.3f} | {doc.metadata['file']:<28} | {doc.page_content[:60]}...")

In [ ]:
plt.figure(figsize=(6, 3))
plt.barh([f"{d.metadata['file']} #{i}" for i, (d, _) in enumerate(scored)], [s for _, s in scored], color="#55A868")
plt.gca().invert_yaxis()
plt.xlabel("Similarity score (higher = closer)")
plt.title("How close is each chunk to the question?")
plt.show()

### Filtering by metadata

Remember the `doc_type` we added? Now it pays off. We can restrict a search to a subset - for example, only product documents. With `InMemoryVectorStore`, the filter is a small Python function.

In [ ]:
product_only = vector_store.similarity_search(
    "What does it cost?", k=2,
    filter=lambda d: d.metadata["doc_type"] == "product",
)

for d in product_only:
    print(d.metadata["file"], "->", d.page_content[:120], "...\n")

### Chroma - a vector store that saves to disk

Embedding costs money and time, so you do not want to redo it every time you restart. Chroma saves everything in a folder. Next time, you just open the folder.

In [ ]:
from langchain_chroma import Chroma

chroma_store = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="novatech_policies",
    persist_directory="chroma_db",           # saved here
    ids=[f"chunk-{i}" for i in range(len(chunks))],   # fixed ids: re-running this cell updates instead of duplicating
)
print("Chunks in Chroma:", chroma_store._collection.count())

In [ ]:
# Later (even after a restart): open the saved store without re-embedding anything
reopened = Chroma(
    collection_name="novatech_policies",
    embedding_function=embeddings,
    persist_directory="chroma_db",
)

reopened.similarity_search("What is the hotel limit in metro cities?", k=1)[0].page_content

Chroma filters use a dictionary instead of a function:

In [ ]:
reopened.similarity_search("rules", k=2, filter={"file": "it_security_policy.txt"})

## 6. Retrievers - the version that plugs into chains

A **retriever** is a simple object with one job: take a question, return relevant Documents. It is a Runnable (Day 3), so it has `invoke` and `batch` and fits straight into a chain. Tomorrow's RAG chain is built around one.

In [ ]:
retriever = vector_store.as_retriever(search_kwargs={"k": 3})

docs = retriever.invoke("What do I do if I lose my laptop?")
for d in docs:
    print("-", d.metadata["file"], ":", d.page_content[:90].replace("\n", " "), "...")

### Better variety with MMR

Plain similarity search sometimes returns three chunks saying nearly the same thing. **MMR** (Maximal Marginal Relevance) picks results that are relevant **and** different from each other. It first fetches `fetch_k` candidates, then chooses the best `k` diverse ones.

In [ ]:
mmr_retriever = vector_store.as_retriever(search_type="mmr", search_kwargs={"k": 3, "fetch_k": 10})

question = "What are the rules about leave?"
print("Similarity:", [d.metadata["file"] for d in retriever.invoke(question)])
print("MMR       :", [d.metadata["file"] for d in mmr_retriever.invoke(question)])

In [ ]:
questions = ["How long is maternity leave?", "Is there a free trial?", "What is the gift limit?"]

for q, found in zip(questions, retriever.batch(questions)):     # retrievers support batch too
    print(f"{q:<32} -> {found[0].metadata['file']}")

## Mini project: NovaTech policy search

Let's wrap everything into a small search tool an HR team could actually use: type a question, get the best matching passages with their source and a relevance score.

In [ ]:
def policy_search(question, k=2):
    print(f"Question: {question}\n")
    for rank, (doc, score) in enumerate(vector_store.similarity_search_with_score(question, k=k), 1):
        page = f", page {doc.metadata['page'] + 1}" if "page" in doc.metadata else ""
        print(f"[{rank}] {doc.metadata['file']}{page}  (score {score:.2f})")
        print("    " + doc.page_content[:250].replace("\n", " ") + "...\n")

In [ ]:
policy_search("Can I accept a gift from a vendor?")

In [ ]:
policy_search("How fast is support for enterprise customers?")

### Your turn

Ask three of your own questions about the NovaTech documents. Try at least one where you use completely different words from the document (for example "time off when I have a baby"). Does semantic search still find the right passage?

In [ ]:
policy_search("time off when I have a baby")

## Common mistakes on Day 5

| What you see | What usually went wrong |
|--------------|-------------------------|
| `UnicodeDecodeError` while loading | Pass `encoding="utf-8"` to the loader |
| Search results are random or poor | Chunks too big or too small; try 300-800 characters with some overlap |
| A new run re-embeds everything (slow, costly) | Use a persistent store (Chroma, FAISS `save_local`) and reopen it |
| Duplicate results in Chroma | You added the same chunks twice without `ids`. Pass fixed ids, or delete `chroma_db/` and rebuild |

## Quick recap

- Every loader returns `Document` objects: `page_content` plus `metadata`.
- Split documents into overlapping chunks; `RecursiveCharacterTextSplitter` is the safe default.
- Embeddings turn meaning into vectors; cosine similarity measures closeness.
- Vector stores search by meaning; metadata filters narrow the search.
- `as_retriever()` gives you a Runnable that returns documents - the bridge to RAG.

## Practice before Day 6

1. Add a new `.txt` policy of your own to `data/policies` and rebuild the store.
2. Compare search results with `chunk_size=200` and `chunk_size=1000` for the same question.
3. Add `"department"` metadata (HR, IT, Finance) to each policy and filter by it.
4. Use `similarity_search_with_score` to print a warning when the best score is low.

**Tomorrow:** we finally connect the retriever to the LLM. The model will answer questions using these documents - and tell you which document the answer came from.